### Feature Ablation: Suspicious Review Detection

This notebook evaluates the contribution of individual features to model performance using Logistic Regression. Each feature is removed one at a time, and the impact on F1-score and recall for the suspicious class is analysed. This helps identify weak, redundant, or potentially harmful features, including those that may introduce data leakage.

In [89]:
!pip install joblib

In [90]:
import os
import json
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, recall_score, precision_score
from scipy.sparse import hstack

In [91]:
# Load data
X_train = joblib.load("shared_suspicious/X_train_features.pkl")
X_test = joblib.load("shared_suspicious/X_test_features.pkl")

y_train = joblib.load("shared_suspicious/y_train.pkl")
y_test = joblib.load("shared_suspicious/y_test.pkl")

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (661918, 9)
Test shape: (165480, 9)


### Baseline Model

In [92]:


model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# full classification report
print("Baseline Classification Report:")
print(classification_report(y_test, y_pred))

# extract key metrics
baseline_f1 = f1_score(y_test, y_pred)
baseline_recall = recall_score(y_test, y_pred)

print("Baseline F1:", baseline_f1)
print("Baseline Recall:", baseline_recall)

Baseline Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97    157528
           1       0.37      0.04      0.07      7952

    accuracy                           0.95    165480
   macro avg       0.66      0.52      0.52    165480
weighted avg       0.93      0.95      0.93    165480

Baseline F1: 0.0664299622036422
Baseline Recall: 0.036468812877263584


### Result Table

In [93]:

ablation_results = pd.DataFrame(columns=[
    "Removed Feature",
    "F1 (Class 1)",
    "Recall (Class 1)"
])

# add baseline row
ablation_results.loc[len(ablation_results)] = [
    "None (Baseline)",
    baseline_f1,
    baseline_recall
]


### Feature ablation

In [94]:

feature_names = X_train.columns.tolist()

for feature in feature_names:
    
    # remove one feature
    X_train_ab = X_train.drop(columns=[feature])
    X_test_ab = X_test.drop(columns=[feature])
    
    # train model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_ab, y_train)
    
    # predict
    y_pred = model.predict(X_test_ab)
    
    # evaluate
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    # store results
    ablation_results.loc[len(ablation_results)] = [
        feature,
        f1,
        recall
    ]

### View Results

In [95]:

ablation_results = ablation_results.sort_values(
    by="F1 (Class 1)", ascending=False
)

ablation_results

,Removed Feature,F1 (Class 1),Recall (Class 1)
8,generic_word_flag,0.066453,0.036469
0,None (Baseline),0.066430,0.036469
7,personal_pronoun_count,0.066216,0.036343
6,exclamation_count,0.065788,0.036092
4,very_short_review,0.064893,0.035589
9,ai_probability_score,0.063003,0.034457
2,extreme_rating_ratio,0.062443,0.034457
5,lexical_diversity,0.060417,0.032948
3,unverified_ratio,0.013786,0.007168
1,review_frequency,0.000000,0.000000


## Feature Ablation Analysis

Feature ablation was conducted to evaluate the contribution of each feature to model performance, with a particular focus on F1-score and recall for the suspicious class.

The results indicate that most features have minimal impact on performance, as removing them resulted in little to no change in F1-score or recall. This suggests that many of the engineered linguistic features (e.g., personal pronoun count, exclamation count, and generic word indicators) provide weak predictive signals and are not effectively contributing to the detection of suspicious reviews.

In contrast, certain behavioural features such as `review_frequency` and `unverified_ratio` were found to be highly influential. The removal of these features led to a significant drop in both F1-score and recall, indicating that the model relies heavily on them for prediction.

Additionally, some features, such as `ai_probability_score` and `lexical_diversity`, were observed to negatively impact performance. Their removal resulted in improved metrics, suggesting that these features introduce noise or redundancy into the model.

Overall, the analysis reveals that the current model is overly dependent on a small subset of behavioural features while failing to leverage more nuanced linguistic patterns. This highlights the need for feature refinement to improve generalisation and enhance the model’s ability to detect suspicious reviews.

### compare with baseline

In [96]:
print("Baseline F1:", baseline_f1)
print("Baseline Recall:", baseline_recall)

Baseline F1: 0.0664299622036422
Baseline Recall: 0.036468812877263584


### Save new features 

In [97]:
# create folder if it doesnt exist
os.makedirs("new_features", exist_ok=True)

selected_features = [
    "review_frequency",
    "unverified_ratio",
    "extreme_rating_ratio",
    "very_short_review"
    
]

with open("new_features/selected_features.json", "w") as f:
    json.dump(selected_features, f)

print("Features saved")

Features saved


### Retrain Logistic model 

In [98]:
# define cleaned feature set from X_train / X_test

selected_features = [
    "review_frequency",
    "unverified_ratio",
    "extreme_rating_ratio",
    "very_short_review"
]

X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]


### Define Parameters

In [99]:
# class weights to test
weight_options = [
    {0:1, 1:1},
    {0:1, 1:2},
    {0:1, 1:3},
    {0:1, 1:4}
]

# thresholds to test
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

# store results
results = []

### Train model

Each model is trained using different class weights. For each trained model, multiple thresholds are tested to evaluate their impact on precision, recall, and F1-score. This allows for identifying the optimal combination of class weights and decision threshold.

In [100]:
for weights in weight_options:
    
    # train the model
    model = LogisticRegression(
        max_iter=1000,
        class_weight=weights
    )
    
    model.fit(X_train_sel, y_train)
    
    # get predicted probabilities
    y_probs = model.predict_proba(X_test_sel)[:, 1]
    
    # test different thresholds
    for threshold in thresholds:
        
        y_pred = (y_probs > threshold).astype(int)
        
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        results.append([
            weights, threshold, precision, recall, f1
        ])


### Result table

The table below shows the performance of different class weight and threshold combinations. The results are sorted by F1-score to identify the best-performing configuration.

In [101]:
results_df = pd.DataFrame(results, columns=[
    "Weights", "Threshold", "Precision", "Recall", "F1"
])

results_df = results_df.sort_values(by="F1", ascending=False)

results_df

,Weights,Threshold,Precision,Recall,F1
6,"{0: 1, 1: 2}",0.4,0.459824,0.656313,0.540773
12,"{0: 1, 1: 3}",0.5,0.452274,0.657822,0.536018
17,"{0: 1, 1: 4}",0.5,0.435802,0.663732,0.526143
5,"{0: 1, 1: 2}",0.3,0.428502,0.665493,0.521328
11,"{0: 1, 1: 3}",0.4,0.426731,0.666499,0.520322
16,"{0: 1, 1: 4}",0.4,0.254487,0.770372,0.382588
10,"{0: 1, 1: 3}",0.3,0.244640,0.790619,0.373659
15,"{0: 1, 1: 4}",0.3,0.235327,0.803194,0.364004
18,"{0: 1, 1: 4}",0.6,0.290611,0.118335,0.168186
19,"{0: 1, 1: 4}",0.7,0.329662,0.105382,0.159710


## Results Interpretation

The results show that combining class weight tuning with threshold adjustment significantly improves model performance compared to earlier configurations.

The best-performing model was achieved using class weights of {0:1, 1:2} with a threshold of 0.4, resulting in the highest F1-score of 0.54. This configuration provides a strong balance between precision (0.46) and recall (0.66), indicating that the model is able to detect a substantial proportion of suspicious reviews while maintaining a relatively low number of false positives.

Other configurations with higher class weights (e.g., {0:1, 1:3} and {0:1, 1:4}) achieved comparable recall but slightly lower precision and F1-scores, suggesting diminishing returns when increasing the weight of the minority class beyond a certain point.

At lower thresholds (e.g., 0.3), recall is higher but precision decreases, indicating more false positives. Conversely, higher thresholds (e.g., 0.6 and 0.7) lead to a sharp drop in recall, making the model too conservative and ineffective for detecting suspicious reviews.

Overall, the results demonstrate that moderate class weighting combined with a lower decision threshold provides the most effective balance for this task.

Based on the evaluation, the final model is selected using class weights of {0:1, 1:2} and a threshold of 0.4.

This configuration achieves the best trade-off between precision and recall, with the highest F1-score among all tested combinations. It significantly improves precision compared to earlier models while maintaining a strong ability to detect suspicious reviews.

This final model is therefore considered the most balanced and practical for the suspicious review detection task.

In [105]:
#create directory if not exits

os.makedirs("model_suspicious", exist_ok=True)

results_df = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Class Weights": ["{0:1, 1:2}"],
    "Threshold": [0.4],
    "Precision": [precision_score(y_test, y_pred)],
    "Recall": [recall_score(y_test, y_pred)],
    "F1 Score": [f1_score(y_test, y_pred)]
})

results_df.to_csv("model_suspicious/final_logistic_results.csv", index=False)

# save model
joblib.dump(model, "model_suspicious/final_logistic_model.pkl")

print("Final model and results saved successfully.")

Final model and results saved successfully.
